# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:

import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"

df = pd.read_csv(url)

# Visualizar as informações para limpeza
print(df.head())
print(df.shape)
print(df.columns)
print(df.info())
print(df.isna().sum())
print(df.duplicated().sum())

In [16]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Verificar os nomes das colunas
df.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type'],
      dtype='str')

In [ ]:
# 1 Novo Dataframe com os valores de clientes inferior a 1.000
df_low_claims_yes = df[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
]

print(df_low_claims_yes.head())
print("Número de clientes:", len(df_low_claims_yes))

In [ ]:
# 2 Analisar prémios, valores e reclamações

df_yes = df[df["response"] == "Yes"]

analysis_policy_gender = df_yes.groupby(
    ["policy_type", "gender"]
).agg(
    average_monthly_premium=("monthly_premium_auto", "mean"),
    average_customer_lifetime_value=("customer_lifetime_value", "mean"),
    average_total_claim_amount=("total_claim_amount", "mean"),
    number_of_customers=("response", "size")
).round(2)

print(analysis_policy_gender)

#

In [ ]:
# 3 Contar clientes por estado

customers_by_state = df["state"].value_counts()

states_over_500 = customers_by_state[
    customers_by_state > 500
]

print(states_over_500)


In [ ]:
# 4 CLV máximo, minimo e mediano por educação
clv_by_education_gender = df.groupby(
    ["education", "gender"]
).agg(
    clv_max=("customer_lifetime_value", "max"),
    clv_min=("customer_lifetime_value", "min"),
    clv_median=("customer_lifetime_value", "median")
).round(2)

print(clv_by_education_gender)

# Conclusão
# O grupo high School M tem CLV maior que os demais grupo, oo grupo Bachelor M é o que tem o menor valor do grupo.
# No grupo Median o grupo High School genero M tem maior valor e o grupo Doctor F tem o menor valor.  


In [33]:
# 5 Tabela de apólice

# Converter a data
df["effective_to_date"] = pd.to_datetime(
    df["effective_to_date"],
    errors="coerce"
)

# Extrair o nome do mês
df["month"] = df["effective_to_date"].dt.month_name()

# Contar os registos por estado e mês
policies_by_state_month = df.groupby(
    ["state", "month"]
).size().unstack(fill_value=0)

# Organizar os meses por ordem cronológica
month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

policies_by_state_month = policies_by_state_month.reindex(
    columns=month_order,
    fill_value=0
)

print(policies_by_state_month)



month       January  February  March  April  May  June  July  August  \
state                                                                  
Arizona        1008       929      0      0    0     0     0       0   
California     1918      1634      0      0    0     0     0       0   
Nevada          551       442      0      0    0     0     0       0   
Oregon         1565      1344      0      0    0     0     0       0   
Washington      463       425      0      0    0     0     0       0   

month       September  October  November  December  
state                                               
Arizona             0        0         0         0  
California          0        0         0         0  
Nevada              0        0         0         0  
Oregon              0        0         0         0  
Washington          0        0         0         0  


In [ ]:
# 6 Os 3 estados com mais apolices

policies_by_state = df.groupby("state")[
    "number_of_policies"
].sum()

top_3_states = policies_by_state.sort_values(
    ascending=False
).head(3)

print("\n--- TOP 3 ESTADOS ---")
print(top_3_states)

# Filtrar os dados para os três estados
df_top_3 = df[
    df["state"].isin(top_3_states.index)
].copy()

# Criar a tabela mensal
top_3_policies_by_month = pd.pivot_table(
    df_top_3,
    index="state",
    columns="month",
    values="number_of_policies",
    aggfunc="sum",
    fill_value=0
)

top_3_policies_by_month = top_3_policies_by_month.reindex(
    columns=range(1, 13),
    fill_value=0
)

top_3_policies_by_month = top_3_policies_by_month.rename(
    columns=month_names
)




In [ ]:
# 1. Contar as respostas Yes e No por canal
response_counts = df.groupby("sales_channel")["response"].value_counts()

print("Contagem de respostas por canal:")
print(response_counts)


# 2. Calcular a taxa de resposta positiva por canal
# Yes = 1 e No = 0

df["positive_response"] = (df["response"] == "Yes") * 1

response_rate = df.groupby("sales_channel")[
    "positive_response"
].mean() * 100

print("\nTaxa de resposta positiva por canal (%):")
print(response_rate.round(2))


# 3. Criar uma tabela com as respostas Yes e No por canal
response_table = df.groupby(
    ["sales_channel", "response"]
).size().unstack(fill_value=0)

print("\nTabela de respostas por canal:")
print(response_table)


# 4. Utilizar melt para transformar colunas em linhas
response_long = response_table.reset_index().melt(
    id_vars="sales_channel",
    var_name="response",
    value_name="number_of_customers"
)

print("\nTabela transformada com melt:")
print(response_long)


# 5. Identificar o canal com maior taxa de resposta
print("\nCanal com maior taxa de resposta positiva:")
print(response_rate.idxmax())

print("Taxa máxima (%):", round(response_rate.max(), 2))
